In [ ]:
# -----------------------------
# 1단계. 기본 집합 및 참여자 데이터 구성
# -----------------------------

T = [1, 2, 3]   # 기간
P = ["P1", "P2", "P3", "P4"]  # SKU 집합

# SKU 정보
skus = {
    "P1": {"name": "shirt_s"},
    "P2": {"name": "shirt_m"},
    "P3": {"name": "shirt_l"},
    "P4": {"name": "shirt_xl"},
}

In [20]:
# -----------------------------
# 1-1. 화주 데이터
# -----------------------------

# ask_price: 화주가 최소한 받고 싶은 단위 가격
# inventory: 기간별 공급 가능 재고
# beta: 화주 쪽 최소 거래비율
#       해당 SKU lot을 사용하려면 최소 beta 비율 이상 거래되어야 함

sellers = {
    "S1": {
        "name": "seller_a",
        "sku_data": {
            "P1": {  # shirt_s
                "inventory": {1: 300, 2: 200, 3: 100},
                "ask_price": {1: 4000, 2: 3900, 3: 3800},
                "beta": 0.30
            },
            "P2": {  # shirt_m
                "inventory": {1: 200, 2: 150, 3: 100},
                "ask_price": {1: 4200, 2: 4100, 3: 4000},
                "beta": 0.20
            }
        }
    },

    "S2": {
        "name": "seller_b",
        "sku_data": {
            "P2": {  # shirt_m
                "inventory": {1: 250, 2: 200, 3: 150},
                "ask_price": {1: 4300, 2: 4200, 3: 4100},
                "beta": 0.30
            },
            "P3": {  # shirt_l
                "inventory": {1: 250, 2: 150, 3: 100},
                "ask_price": {1: 4400, 2: 4300, 3: 4200},
                "beta": 0.30
            }
        }
    },

    "S3": {
        "name": "seller_c",
        "sku_data": {
            "P3": {  # shirt_l
                "inventory": {1: 150, 2: 200, 3: 200},
                "ask_price": {1: 4500, 2: 4400, 3: 4300},
                "beta": 0.40
            },
            "P4": {  # shirt_xl
                "inventory": {1: 500, 2: 400, 3: 300},
                "ask_price": {1: 4100, 2: 4000, 3: 3900},
                "beta": 0.10
            }
        }
    }
}

In [9]:
# 구매자 데이터
# order_value 또는 budget은 주문 전체 기준 지불의향
# item별 unit_bid는 SKU 단위 지불의향

buyers = {
    "B1": {
        "name": "Retailer_A",
        "orders": {
            "O1": {
                "request_time": 1,
                "due_time": 2,
                "budget": 2500000,
                "split_allowed": True,
                "delay_allowed": True,
                "items": {
                    "P1": {"demand": 300, "unit_bid": 6000},
                    "P2": {"demand": 200, "unit_bid": 4500},
                    "P4": {"demand": 300, "unit_bid": 1800},
                }
            }
        }
    },

    "B2": {
        "name": "Wholesaler_B",
        "orders": {
            "O2": {
                "request_time": 1,
                "due_time": 1,
                "budget": 1800000,
                "split_allowed": False,
                "delay_allowed": False,
                "items": {
                    "P1": {"demand": 200, "unit_bid": 6200},
                    "P3": {"demand": 150, "unit_bid": 7000},
                }
            }
        }
    }
}

In [10]:
# 운송사 / 배송 옵션 데이터
# cost는 단위당 운송+fulfillment 비용으로 단순화

transporters = {
    "K1": {
        "name": "3PL_A",
        "cap_weight": {1: 300, 2: 300, 3: 300},
        "cap_volume": {1: 5.0, 2: 5.0, 3: 5.0},
        "unit_cost": 500
    },
    "K2": {
        "name": "3PL_B",
        "cap_weight": {1: 200, 2: 200, 3: 200},
        "cap_volume": {1: 4.0, 2: 4.0, 3: 4.0},
        "unit_cost": 650
    }
}

In [11]:
# -----------------------------
# 2단계. 주문 line별 alpha 설정
# -----------------------------

# alpha 의미:
# alpha = 1.0  : 전량 이행 필수
# 0 < alpha < 1: 부분 이행 허용
# alpha = 0.0  : 선택 품목, 없어도 주문 수락 가능

buyers["B1"]["orders"]["O1"]["items"]["P1"]["alpha"] = 1.0   # 선크림은 전량 필수
buyers["B1"]["orders"]["O1"]["items"]["P2"]["alpha"] = 0.7   # 토너는 70% 이상이면 가능
buyers["B1"]["orders"]["O1"]["items"]["P4"]["alpha"] = 0.0   # 마스크팩은 선택 품목

buyers["B2"]["orders"]["O2"]["items"]["P1"]["alpha"] = 1.0
buyers["B2"]["orders"]["O2"]["items"]["P3"]["alpha"] = 0.8

In [12]:
# 주문 line 데이터 생성
order_lines = []

for b, bdata in buyers.items():
    for o, odata in bdata["orders"].items():
        for p, item in odata["items"].items():
            order_lines.append({
                "buyer": b,
                "order": o,
                "sku": p,
                "demand": item["demand"],
                "alpha": item["alpha"],
                "unit_bid": item["unit_bid"],
                "request_time": odata["request_time"],
                "due_time": odata["due_time"],
                "budget": odata["budget"],
                "split_allowed": odata["split_allowed"],
                "delay_allowed": odata["delay_allowed"]
            })

order_lines

[{'buyer': 'B1',
  'order': 'O1',
  'sku': 'P1',
  'demand': 300,
  'alpha': 1.0,
  'unit_bid': 6000,
  'request_time': 1,
  'due_time': 2,
  'budget': 2500000,
  'split_allowed': True,
  'delay_allowed': True},
 {'buyer': 'B1',
  'order': 'O1',
  'sku': 'P2',
  'demand': 200,
  'alpha': 0.7,
  'unit_bid': 4500,
  'request_time': 1,
  'due_time': 2,
  'budget': 2500000,
  'split_allowed': True,
  'delay_allowed': True},
 {'buyer': 'B1',
  'order': 'O1',
  'sku': 'P4',
  'demand': 300,
  'alpha': 0.0,
  'unit_bid': 1800,
  'request_time': 1,
  'due_time': 2,
  'budget': 2500000,
  'split_allowed': True,
  'delay_allowed': True},
 {'buyer': 'B2',
  'order': 'O2',
  'sku': 'P1',
  'demand': 200,
  'alpha': 1.0,
  'unit_bid': 6200,
  'request_time': 1,
  'due_time': 1,
  'budget': 1800000,
  'split_allowed': False,
  'delay_allowed': False},
 {'buyer': 'B2',
  'order': 'O2',
  'sku': 'P3',
  'demand': 150,
  'alpha': 0.8,
  'unit_bid': 7000,
  'request_time': 1,
  'due_time': 1,
  'budget'

In [15]:
for line in order_lines:
    line["min_qty"] = line["alpha"] * line["demand"]

order_lines

[{'buyer': 'B1',
  'order': 'O1',
  'sku': 'P1',
  'demand': 300,
  'alpha': 1.0,
  'unit_bid': 6000,
  'request_time': 1,
  'due_time': 2,
  'budget': 2500000,
  'split_allowed': True,
  'delay_allowed': True,
  'min_qty': 300.0},
 {'buyer': 'B1',
  'order': 'O1',
  'sku': 'P2',
  'demand': 200,
  'alpha': 0.7,
  'unit_bid': 4500,
  'request_time': 1,
  'due_time': 2,
  'budget': 2500000,
  'split_allowed': True,
  'delay_allowed': True,
  'min_qty': 140.0},
 {'buyer': 'B1',
  'order': 'O1',
  'sku': 'P4',
  'demand': 300,
  'alpha': 0.0,
  'unit_bid': 1800,
  'request_time': 1,
  'due_time': 2,
  'budget': 2500000,
  'split_allowed': True,
  'delay_allowed': True,
  'min_qty': 0.0},
 {'buyer': 'B2',
  'order': 'O2',
  'sku': 'P1',
  'demand': 200,
  'alpha': 1.0,
  'unit_bid': 6200,
  'request_time': 1,
  'due_time': 1,
  'budget': 1800000,
  'split_allowed': False,
  'delay_allowed': False,
  'min_qty': 200.0},
 {'buyer': 'B2',
  'order': 'O2',
  'sku': 'P3',
  'demand': 150,
  'alp

In [16]:
net_unit = buyer_price - seller_price - transporter_cost

if net_unit >= 0:
    valid_tuple.append((seller, buyer, transporter, time))

NameError: name 'buyer_price' is not defined